In [1]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, polars as pl
import lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression

ROOT = Path.cwd().parent
P = ROOT / "data" / "processed"
tr = pl.read_parquet(P / "train_features.parquet").to_pandas()
va = pl.read_parquet(P / "validation_features.parquet").to_pandas()
for d in (tr, va):
    d["sound_log"] = np.log1p(d["sound_freq"])

CAT = ["country", "language", "type"]
NUM = ["hour", "weekday", "duration_ms", "caption_len", "n_hashtags",
       "n_mentions", "n_onscreen", "is_ad", "is_shop_video", "sound_log"]
FEATS = CAT + NUM
y_tr, y_va = tr["is_viral"], va["is_viral"]
cats = {c: sorted(tr[c].unique()) for c in CAT}

def prep(d):
    X = d[FEATS].copy().reset_index(drop=True)
    for c in CAT:
        X[c] = pd.Categorical(X[c].where(X[c].isin(cats[c])), categories=cats[c])
    return X

def fit_gbm(Xa, Xb):
    m = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.03, num_leaves=31,
        min_child_samples=50, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        random_state=42, verbose=-1)
    m.fit(Xa, y_tr, eval_set=[(Xb, y_va)], eval_metric="average_precision",
          callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    return m

def report(name, y, p):
    k = int(len(y) * 0.10); top = np.argsort(-p)[:k]
    print(f"{name}: PR-AUC {average_precision_score(y, p):.3f} | ROC-AUC {roc_auc_score(y, p):.3f} | recall top 10% {y.values[top].sum()/y.sum():.3f}")

def paired_gain(p_new, p_old, n=500):
    rng = np.random.default_rng(0); out = []
    for _ in range(n):
        i = rng.integers(0, len(y_va), len(y_va)); yy = y_va.values[i]
        out.append(average_precision_score(yy, p_new[i]) - average_precision_score(yy, p_old[i]))
    lo, hi = np.percentile(out, [2.5, 97.5])
    print(f"gain {np.mean(out):.3f}, 95% interval [{lo:.3f}, {hi:.3f}]")

In [2]:
lab_tr = pl.read_parquet(P / "train_labeled.parquet").select(["video_id", "viral_B"]).to_pandas()
lab_va = pl.read_parquet(P / "validation_labeled.parquet").select(["video_id", "viral_B"]).to_pandas()
yB_tr = tr[["video_id"]].merge(lab_tr, on="video_id", how="left")["viral_B"].astype(int)
yB_va = va[["video_id"]].merge(lab_va, on="video_id", how="left")["viral_B"].astype(int)
print("positive rate B:", yB_tr.mean().round(3), yB_va.mean().round(3))

m = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.03, num_leaves=31, min_child_samples=50,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=42, verbose=-1)
m.fit(prep(tr), yB_tr, eval_set=[(prep(va), yB_va)], eval_metric="average_precision",
      callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
pB = m.predict_proba(prep(va))[:, 1]
print("PR-AUC label B:", round(average_precision_score(yB_va, pB), 3), "| random:", round(yB_va.mean(), 3))
imp = pd.Series(m.booster_.feature_importance("gain"), index=FEATS)
print((imp / imp.sum()).sort_values(ascending=False).round(3))

positive rate B: 0.05 0.05
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[59]	valid_0's average_precision: 0.11447	valid_0's binary_logloss: 0.186976
PR-AUC label B: 0.114 | random: 0.05
caption_len      0.354
duration_ms      0.218
language         0.076
n_hashtags       0.075
hour             0.072
n_onscreen       0.047
weekday          0.041
is_ad            0.041
country          0.038
n_mentions       0.018
sound_log        0.012
is_shop_video    0.007
type             0.002
dtype: float64


## Conclusion: robustness to the viral definition

With the stricter label (top 5% of views per month), LightGBM reaches PR-AUC 0.114 against a
random baseline of 0.05, a ratio of 2.3 (versus 2.0 for the top 10% label). The most important
features are the same and in the same order (caption length, duration, language, hashtags, hour).
The conclusions therefore do not depend on the exact threshold.
**Limit:** the stricter label has about half as many positives, so this estimate is noisier.